# Agentic RFP Evaluation & Supplier Ranking

This notebook demonstrates the core evaluation pipeline from the mini-project brief. It creates four **fictional** proposal PDFs, extracts their text, evaluates every active SQLite criterion, validates scorecards, calculates peer benchmarks and the weighted Peer Performance Index (PPI), ranks suppliers deterministically, and persists one run to SQLite.

The **Demo** mode runs without an AI key and uses a transparent keyword heuristic; it is not an LLM assessment. **Live** mode calls an OpenAI-compatible chat model and requires a key. The complete Streamlit application is in `app.py`.

Run cells from top to bottom. Install project dependencies first if needed: `python -m pip install -r requirements.txt`. Keep API keys out of notebook source and version control.

In [ ]:
from pathlib import Path
import json
import sys

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'evaluation.py').exists():
    PROJECT_DIR = Path(r'C:\Users\gupta\source\repos\agentic-rfp-evaluation')
if not (PROJECT_DIR / 'evaluation.py').exists():
    raise FileNotFoundError('Open this notebook from the agentic-rfp-evaluation project folder.')
sys.path.insert(0, str(PROJECT_DIR))

from database import DB_PATH, create_run, get_criteria, initialize_database, persist_results
from documents import create_sample_proposals, extract_pdf_text
from evaluation import calculate_scores, demo_evaluation, evaluate_with_llm, normalize_evaluation

initialize_database()
criteria = get_criteria(active_only=True)
assert criteria, 'No active criteria found.'
assert abs(sum(item['weight'] for item in criteria) - 100) < 0.01, 'Active weights must total 100%.'
print(f'Project: {PROJECT_DIR}')
print(f'Database: {DB_PATH}')
print(f'Active criteria: {len(criteria)}; total weight: {sum(item["weight"] for item in criteria):g}%')

In [ ]:
# Generate the four brief-required fictional supplier proposals (three pages each).
sample_dir = PROJECT_DIR / 'sample_proposals'
proposals = create_sample_proposals(sample_dir)
[(item['name'], item['path'].name, item['path'].stat().st_size) for item in proposals]

## Choose the evaluation mode

Keep `MODE` set to `"demo"` to test locally without credentials. For actual LLM scoring, change it to `"live"`; the next cell will prompt for the key without echoing or storing it in the notebook. Live model calls may incur provider charges.

In [ ]:
MODE = 'demo'  # Change to 'live' to call the configured LLM provider.
MODEL = 'gpt-4o-mini'
api_key = None
if MODE == 'live':
    import getpass
    api_key = getpass.getpass('LLM API key (input is hidden): ')
    if not api_key.strip():
        raise ValueError('A key is required in live mode. Switch MODE to demo to run without one.')
elif MODE != 'demo':
    raise ValueError("MODE must be either 'demo' or 'live'.")

scorecards = []
warnings = []
for proposal in proposals:
    text = extract_pdf_text(proposal['path'])
    raw = (
        demo_evaluation(proposal['name'], text, criteria)
        if MODE == 'demo'
        else evaluate_with_llm(proposal['name'], text, criteria, api_key, MODEL)
    )
    scorecard, validation_warnings = normalize_evaluation(raw, criteria, proposal['name'])
    scorecard.update({
        'submission_date': proposal['date'],
        'experience_rating': proposal['experience'],
        'source_file': proposal['path'].name,
        'evaluation_mode': MODE,
    })
    scorecards.append(scorecard)
    warnings.extend(validation_warnings)
print(f'Evaluated {len(scorecards)} fictional suppliers using {MODE} mode.')
print(f'Validation warnings: {len(warnings)}')

In [ ]:
ranked_suppliers, ranking_warnings = calculate_scores(scorecards, criteria)
warnings.extend(ranking_warnings)
run_id = create_run()
persist_results(run_id, ranked_suppliers)
print(f'Completed and persisted run: {run_id}')
print(f'Warnings: {warnings}')

In [ ]:
# Display a compact leaderboard and inspect one evidence-backed scorecard.
import pandas as pd
leaderboard = pd.DataFrame([
    {key: supplier[key] for key in ('final_rank', 'supplier_name', 'absolute_score', 'ppi', 'submission_date', 'experience_rating')}
    for supplier in ranked_suppliers
])
display(leaderboard)
display(pd.DataFrame(ranked_suppliers[0]['criteria']))

In [ ]:
# Export the same complete run shape that the Streamlit app offers for download.
export = {
    'rfp_run_id': run_id,
    'warnings': warnings,
    'ranking_rule': 'PPI descending, submission date ascending, experience rating descending, supplier name ascending',
    'suppliers': ranked_suppliers,
}
export_path = PROJECT_DIR / f'rfp_run_{run_id}.json'
export_path.write_text(json.dumps(export, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'JSON export saved to {export_path}')

## Scoring rules

- Absolute weighted score: $\sum_i (score_i / max_i) \times weight_i$ (out of 100).
- Criterion benchmark: the highest valid score for that criterion across this batch.
- Criterion gap: supplier score minus benchmark; the benchmark leader has a gap of zero.
- Relative performance: $score / benchmark \times 100$; when the benchmark is zero, this implementation sets relative performance to zero and records a warning.
- PPI: the weighted average of criterion relative-performance percentages.
- Stable rank order: PPI descending, earlier submission date, higher experience rating, then supplier name ascending. Python assigns sequential ranks only after sorting.

Demo scores are heuristic examples, not procurement recommendations. Review the extracted evidence and risks before making a decision.